# 01 · Ollama + Pinggy (Server)

Run this notebook in **Colab**. It follows the original simple Ollama → Pinggy workflow for Practice 03 RAG Chat.

**Only four steps:** install Ollama → start server → pull models (`embeddinggemma` & `qwen3:4b`) → open Pinggy tunnel.

### 1. Install Ollama

In [ ]:
# Install zstd
!apt-get update -qq
!apt-get install -y -qq zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Check installation
!ollama --version

### 2. Start Ollama locally

In [ ]:
import os, subprocess, time, requests

env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
log = open("/tmp/ollama.log", "a")
process = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Check /tmp/ollama.log")
print("Ollama is running on localhost:11434")

### 3. Tải các mô hình (embeddinggemma & qwen3:4b) và kiểm tra API

In [ ]:
import os, subprocess, time, requests

# Đảm bảo ollama serve đang chạy
try:
    requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
except Exception:
    print("⏳ Đang khởi động máy chủ Ollama ngầm...")
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"
    log = open("/tmp/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(30):
        try:
            if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:
                break
        except Exception:
            pass
        time.sleep(1)
    print("✅ Ollama server đã khởi động thành công!")

# Pull embedding model
!ollama pull embeddinggemma

# Pull generation model (qwen3:4b)
!ollama pull qwen3:4b

print("⏳ Đang kiểm tra kết nối tới các model...")

try:
    res_embed = requests.post(
        "http://127.0.0.1:11434/api/embed",
        json={"model": "embeddinggemma", "input": "Xin chào!"},
        timeout=180,
    )
    res_embed.raise_for_status()
    print("✅ Model 1/2: embeddinggemma sẵn sàng!")
except Exception as e:
    print("⚠️ Model embeddinggemma đang được chuẩn bị:", e)

try:
    res_gen = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={"model": "qwen3:4b", "prompt": "Xin chào!", "stream": False},
        timeout=180,
    )
    res_gen.raise_for_status()
    print("✅ Model 2/2: qwen3:4b sẵn sàng!")
except Exception as e:
    print("⚠️ Model qwen3:4b đang được chuẩn bị:", e)


### 4. Tự động mở đường hầm Pinggy và hiển thị Public URL

Chạy cell bên dưới, script sẽ tự động kết nối Pinggy và in ra đường link HTTPS công khai.
Bạn chỉ cần copy đường link đó và dán vào file .env trên máy cá nhân.

In [ ]:
import os, requests
# Đảm bảo Ollama server đang chạy
try:
    requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
except Exception:
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"
    log = open("/tmp/ollama.log", "a")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)
    time.sleep(3)

import subprocess
import time
import re

print("⏳ Đang kết nối tới Pinggy để lấy Public HTTPS URL...")

ssh_cmd = [
    "ssh",
    "-p", "443",
    "-R0:localhost:11434",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ServerAliveInterval=30",
    "a.pinggy.io",
    "u:Host:localhost:11434"
]

tunnel = subprocess.Popen(
    ssh_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
for line in tunnel.stdout:
    line_clean = line.strip()
    if "dashboard.pinggy.io" in line_clean:
        continue
    match = re.search(r"https://[a-zA-Z0-9\-\.]+(?:pinggy|link|net)[a-zA-Z0-9\-\.]*", line_clean)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "=" * 65)
    print("🎉 SERVER OLLAMA ĐÃ SẴN SÀNG HOẠT ĐỘNG!")
    print(f"🔗 Public URL: {public_url}")
    print("\n👉 BƯỚC TIẾP THEO: Dán dòng này vào file .env trên máy của bạn:")
    print(f"   OLLAMA_BASE_URL={public_url}")
    print("=" * 65)
    print("\n⚠️ GIỮ CELL NÀY CHẠY LIÊN TỤC để duy trì server hoạt động...")
    try:
        while True:
            time.sleep(10)
    except KeyboardInterrupt:
        print("\n🛑 Đã dừng server Pinggy.")
        tunnel.terminate()
else:
    print("❌ Chưa lấy được URL Pinggy. Bạn có thể mở xterm bằng cell dự phòng bên dưới.")


### 5. (Tùy chọn) Mở Terminal xterm dự phòng nếu bước trên gặp sự cố

In [ ]:
%pip -q install colab-xterm
%load_ext colabxterm
%xterm
# Sau khi mở terminal, gõ lệnh:
# ssh -p 443 -R0:localhost:11434 -t qr@free.pinggy.io "u:Host:localhost:11434"
